In [ ]:
import ast
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from ipywidgets import AppLayout, Button, Layout, HBox, VBox
from IPython.display import display, IFrame
from bokeh.plotting import figure
from bokeh.io import output_file, save
from bokeh.models import ColumnDataSource
from itables import show as itable_show

# Configure Bokeh output
output_file("bokeh_plot.html")


class Dashboard:
    """
    A  interactive dashboard for data visualisation and filtering.

    This class leverages ipywidgets for the UI, pandas for data manipulation,
    and both Matplotlib and Bokeh for rendering plots. It allows users to
    filter data by time and categories dynamically.

    Attributes:
        original_df (pd.DataFrame): The source dataset.
        filtered_df (pd.DataFrame): The dataset after user filters are applied.
        ui (VBox): The master container for the dashboard layout.
    """

    def __init__(self, df):
        """
        Initialises the Dashboard with a specific dataset.

        Args:
            df (pd.DataFrame): The DataFrame to be used for analysis. 
                               Must contain 'seen' and 'amount' columns.
        """
        self.original_df = df
        self.min_time_from_df = pd.to_datetime(self.original_df['seen'].min())
        self.max_time_from_df = pd.to_datetime(self.original_df['seen'].max())

        # Store empty df but keep columns of original
        self.filtered_df = pd.DataFrame(columns=df.columns)

        self.fields_to_plot = self.get_fields_to_plot(self.original_df)
        self.warning_flag = False
        self.default_min_count_to_plot = 5
        self.default_max_values_to_plot = 5

        self._create_ui()
        self._wire_up_events()

    def _create_ui(self):
        """
        Create and layout all UI components for the dashboard.
        """
        # Dashboard title
        self.dashboard_title = widgets.HTML(
            value="<div style='text-align: center; width: 100%;'>"
                  "<h1>My Dashboard Title</h1></div>"
        )

        # Date/time pickers
        self.start_datetime_picker = widgets.NaiveDatetimePicker(
            description='Pick a Time', value=self.min_time_from_df
        )
        self.end_datetime_picker = widgets.NaiveDatetimePicker(
            description='Pick a Time', value=self.max_time_from_df
        )

        # Radio button selection for plot types
        self.plot_picker = widgets.RadioButtons(
            options=['A', 'B', 'C'],
            value='A',
            layout=Layout(justify_items='center', align_items='center')
        )

        # Selection dropdown for field filter
        self.field_select = widgets.Select(
            options=self.fields_to_plot,
            value=None,
            description=' ',
            layout=Layout(justify_items='center', align_items='center')
        )

        # Main control rows
        user_options = HBox([
            VBox([
                widgets.HTML("<div style='text-align: center;'><h3>Plot</h3></div>"),
                self.plot_picker
            ]),
            VBox([
                widgets.HTML("<div style='text-align: center;'><h3>Start/End Time</h3></div>"),
                self.start_datetime_picker,
                self.end_datetime_picker
            ]),
            VBox([
                widgets.HTML("<div style='text-align: center;'><h3>Field to plot</h3></div>"),
                self.field_select
            ])
        ])

        filter_options_output = self.filter_options()

        # Plotting configuration widgets
        self.min_count_to_plot = widgets.BoundedIntText(
            value=self.default_min_count_to_plot,
            min=0, max=10, step=1,
            layout=Layout(width='100px')
        )

        self.max_values_to_plot = widgets.BoundedIntText(
            value=self.default_max_values_to_plot,
            min=0, max=10, step=1,
            layout=Layout(width='100px')
        )

        plotting_options = VBox(
            [
                HBox([widgets.Label("min_count_to_plot"), self.min_count_to_plot]),
                HBox([widgets.Label("max_values_to_plot"), self.max_values_to_plot])
            ],
            layout=Layout(display='flex', width='auto', justify_content='center')
        )

        # Accordion for collapsible sections
        self.filter_accordion = widgets.Accordion(
            children=[filter_options_output, plotting_options]
        )
        self.filter_accordion.set_title(0, 'Filter Options')
        self.filter_accordion.set_title(1, 'Plotting Options')

        user_options_with_accordion = VBox([user_options, self.filter_accordion])

        # Action buttons
        self.go_button = Button(
            description='Go', button_style='success',
            layout=Layout(height='50px', width='300px')
        )
        self.reset_button = Button(
            description='Reset', button_style='warning',
            layout=Layout(height='50px', width='300px')
        )

        go_and_reset_buttons = HBox(
            [self.go_button, self.reset_button],
            layout=Layout(display='flex', width='auto', justify_content='center')
        )

        self.info_message_to_user = widgets.HTML(value="")
        self.plot_df_output = widgets.Output()
        self.plot_output = widgets.Output()
        self.df_output = widgets.Output()

        self.output_tab = widgets.Tab()
        self.output_tab.children = [self.plot_output, self.df_output]
        self.output_tab.set_title(0, 'Plot')
        self.output_tab.set_title(1, 'DataFrame')

        # Assemble UI
        self.ui = VBox([
            AppLayout(
                header=self.dashboard_title,
                center=user_options_with_accordion,
                footer=VBox([go_and_reset_buttons, self.info_message_to_user]),
                layout=Layout(border='3px solid black')
            ),
            self.plot_df_output
        ])

    def _wire_up_events(self):
        """
        Attach button click events to their respective callback methods.
        """
        self.go_button.on_click(self.on_go_click)
        self.reset_button.on_click(self.on_reset_click)

    def on_go_click(self, b):
        """
        Handle the 'Go' button click. Triggers data filtering and plot generation.

        Args:
            b (widgets.Button): The button instance that triggered the event.
        """
        self.warning_flag = False
        self.info_message_to_user.value = "<b>Loading...</b>"

        self.plot_output.clear_output()
        self.plot_df_output.clear_output()
        self.df_output.clear_output()

        with self.plot_output:
            if self.start_datetime_picker.value is None or self.end_datetime_picker.value is None:
                self.display_message_to_user("Please select both dates")
            else:
                self.filter_df_by_date_time()
                if self.filtered_df.empty:
                    self.display_message_to_user("No events in range; change date parameters")

            self.apply_user_filters()

            if self.filtered_df.empty and not self.warning_flag:
                self.display_message_to_user("No events in range; change filter options")

            if self.field_select.value is None and not self.warning_flag:
                self.display_message_to_user("Please select a Field To Plot")
            elif not self.warning_flag:
                if self.plot_picker.value == 'A':
                    self.plot_a()
                elif self.plot_picker.value == 'B':
                    self.plot_b()
                elif self.plot_picker.value == 'C':
                    self.plot_c()

        with self.df_output:
            if not self.warning_flag:
                itable_show(self.filtered_df, buttons=["csv", "excel"])

        with self.plot_df_output:
            display(self.output_tab)

    def on_reset_click(self, b):
        """
        Handle the 'Reset' button click. Reverts UI widgets to default values.

        Args:
            b (widgets.Button): The button instance that triggered the event.
        """
        self.warning_flag = False
        with self.plot_df_output:
            self.plot_df_output.clear_output()
            self.start_datetime_picker.value = self.min_time_from_df
            self.end_datetime_picker.value = self.max_time_from_df
            self.plot_picker.value = 'A'
            self.field_select.value = None
            self.filter_accordion.selected_index = None

            children = list(self.filter_accordion.children)
            children[0] = self.filter_options()
            self.filter_accordion.children = tuple(children)

            self.max_values_to_plot.value = self.default_max_values_to_plot
            self.min_count_to_plot.value = self.default_min_count_to_plot
            self.info_message_to_user.value = ""

    def filter_df_by_date_time(self):
        """
        Filter the source DataFrame based on the user-selected date range.
        """
        self.filtered_df = self.original_df.copy(deep=True)
        self.filtered_df["seen"] = pd.to_datetime(self.filtered_df['seen'])
        self.filtered_df = self.filtered_df[
            self.filtered_df["seen"].between(
                self.start_datetime_picker.value, 
                self.end_datetime_picker.value
            )
        ]

    def plot_a(self):
        """
        Generate a Matplotlib bar chart based on the sum of amounts per field.
        """
        grouped = self.filtered_df.groupby(by=self.field_select.value)['amount'].sum().reset_index()
        plt.bar(grouped[self.field_select.value], grouped['amount'])
        plt.show()

    def plot_b(self):
        """
        Generate an interactive Bokeh bar chart and display it via an IFrame.
        """
        grouped = self.filtered_df.groupby(by=self.field_select.value)['amount'].sum().reset_index()
        p = figure(
            x_range=grouped[self.field_select.value],
            height=800, width=800,
            tools="pan,wheel_zoom,box_zoom,reset,save"
        )
        source = ColumnDataSource(grouped)
        p.vbar(x=self.field_select.value, top='amount', width=1, source=source)
        save(p)
        display(IFrame("bokeh_plot.html", width=600, height=1000))

    def plot_c(self):
        """
        Placeholder for plot type C implementation.
        """
        pass

    def display_message_to_user(self, text_to_display):
        """
        Render a specific message (usually a warning) in the UI for the user.

        Args:
            text_to_display (str): The message content to show in red HTML.
        """
        self.warning_flag = True
        display(widgets.HTML(f"<p style='color:red;'>{text_to_display}</p>"))

    def apply_user_filters(self):
        """
        Iterate through active filter buttons and apply their logic to the DataFrame.
        """
        for btn in self.filters_hbox.children:
            desc = btn.description.strip()
            if " NOT IN " in desc:
                col, val_str = desc.split(" NOT IN ", 1)
                values = ast.literal_eval(val_str)
                self.filtered_df = self.filtered_df[~self.filtered_df[col].isin(values)]
            elif " IN " in desc:
                col, val_str = desc.split(" IN ", 1)
                values = ast.literal_eval(val_str)
                self.filtered_df = self.filtered_df[self.filtered_df[col].isin(values)]

    def get_fields_to_plot(self, df):
        """
        Extract a list of columns eligible for categorical plotting.

        Args:
            df (pd.DataFrame): The DataFrame to check for columns.

        Returns:
            list: Column names excluding technical fields like 'amount' and 'seen'.
        """
        to_remove = ['amount', 'seen']
        return [x for x in list(self.original_df.columns) if x not in to_remove]

    def filter_options(self):
        """
        Generate the advanced categorical filtering widget interface.

        Returns:
            widgets.Output: The UI container for adding/removing data filters.
        """
        not_include_in_filter = ['amount', 'seen']
        to_filter = [
            x for x in self.original_df.select_dtypes(include=['object', 'category']).columns.tolist() 
            if x not in not_include_in_filter
        ]

        filter_options_output = widgets.Output()
        filter_output = widgets.Output()

        col_dropdown = widgets.Dropdown(options=to_filter, description="Column:")

        def create_filter_widget(col):
            """Internal helper to create a multi-select widget for a column."""
            return widgets.SelectMultiple(
                options=self.original_df[col].unique(),
                value=list(self.original_df[col].unique()),
                description=col
            )

        def update_filter_widget(change):
            """Internal callback to refresh the multi-select widget on dropdown change."""
            with filter_output:
                filter_output.clear_output()
                widget = create_filter_widget(change['new'])
                display(widget)
                filter_output.widget = widget

        col_dropdown.observe(update_filter_widget, names='value')

        include_btn = Button(description="Keep matching", button_style='success', layout=Layout(width='auto'))
        exclude_btn = Button(description="Remove matching", button_style='danger', layout=Layout(width='auto'))
        reset_btn = Button(description="Reset", button_style='warning', layout=Layout(width='auto'))

        buttons_box = HBox([include_btn, exclude_btn, reset_btn])
        self.filters_hbox = HBox(layout=Layout(flex_flow='row wrap', gap='5px'))

        def find_button(substring):
            """Helper to locate an existing filter button by its description."""
            return next((b for b in self.filters_hbox.children if substring in b.description), None)

        def add_include_filter(btn):
            """Logic to add an 'IN' filter button to the UI."""
            desc = f"{col_dropdown.value} IN {list(filter_output.widget.value)}"
            existing = find_button(f"{col_dropdown.value} IN")
            if existing:
                existing.description = desc
            else:
                new_btn = Button(description=desc, button_style='success', disabled=True, layout=Layout(width='auto'))
                self.filters_hbox.children = list(self.filters_hbox.children) + [new_btn]

        def add_exclude_filter(btn):
            """Logic to add a 'NOT IN' filter button to the UI."""
            desc = f"{col_dropdown.value} NOT IN {list(filter_output.widget.value)}"
            existing = find_button(f"{col_dropdown.value} NOT IN")
            if existing:
                existing.description = desc
            else:
                new_btn = Button(description=desc, button_style='danger', disabled=True, layout=Layout(width='auto'))
                self.filters_hbox.children = list(self.filters_hbox.children) + [new_btn]

        def reset_filters(btn):
            """Clears all active filter buttons and resets the multi-select."""
            self.filters_hbox.children = []
            with filter_output:
                filter_output.clear_output()
                widget = create_filter_widget(col_dropdown.value)
                display(widget)
                filter_output.widget = widget

        include_btn.on_click(add_include_filter)
        exclude_btn.on_click(add_exclude_filter)
        reset_btn.on_click(reset_filters)

        with filter_options_output:
            display(VBox([
                col_dropdown, filter_output, buttons_box,
                widgets.HTML("<hr style='border:1px solid gray; width:100%'>"),
                self.filters_hbox
            ]))

        update_filter_widget({'new': col_dropdown.value})
        return filter_options_output

    def show(self):
        """
        Display the master dashboard UI in the Jupyter Notebook output cell.
        """
        display(self.ui)

In [ ]:
import pandas as pd

# Example DataFrame
df = pd.read_csv('test.csv')
# Create dashboard
dash = Dashboard(df)

# Show it
dash.show()